# Klasifikasi Kondisi Paket dari Foto — Pipeline Lengkap
**DATASCAPE 2026 · HIMA DS Telkom University**

Notebook ini membangun model *computer vision* yang mengklasifikasikan foto paket ke dalam 3 kelas:

| Kelas | Arti |
|---|---|
| **Aman** | Paket/dus utuh, mulus, layak kirim |
| **Rusak** | Paket/dus penyok, sobek, basah, atau hancur |
| **Lainnya** | Kemasan non-standar / *out-of-distribution* (kemasan makanan, kantong plastik, paper bag) |

Metrik lomba: **Macro-F1** (rata-rata F1 ketiga kelas tanpa bobot).

## Alur kerja (8 tahap)

| Tahap | Isi | Bagian notebook |
|---|---|---|
| 0 | Persiapan lingkungan & unduh data dari Hugging Face | §0–§1 |
| 1 | Split train/val (stratified, seed 42) | §2 |
| 2 | Audit data (md5, phash, duplikat, label konflik, artefak augmentasi, *val bersih*) | §3 |
| 3 | Membaca gambar dengan aman + cache + letterbox | §4 |
| 4 | Embedding *foundation model* beku (CLIP ViT-B/16 + DINOv2 ViT-S/14) | §5 |
| 5 | Classifier non-linear di atas embedding (MLP 3 seed) | §6 |
| 6 | Putaran 2: CLIP multi-crop 448 px + MLP | §7 |
| 7 | Fine-tune CNN (ConvNeXt-tiny putaran 1 & 2) | §8 |
| 8 | Ensemble log-prob berbobot + bias kelas + rata-rata foto kembar → `submission.csv` | §9–§10 |

Eksperimen pembanding (zero-shot CLIP, Logistic Regression, kNN, SVM, CLIP ViT-L/14, EfficientNetV2 + focal loss, YOLO11-cls) juga tersedia di §11, **tetapi mati secara default** karena tidak masuk ensemble final.

## Cara menjalankan di Google Colab
1. `Runtime → Change runtime type → GPU` (T4 sudah cukup; L4/A100 lebih cepat).
2. `Runtime → Run all`. Saat diminta, izinkan akses Google Drive.
3. Semua hasil (split, audit, embedding, checkpoint, prediksi) disimpan di `MyDrive/package-qc/`. Bila sesi Colab putus, cukup **Run all** lagi: tahap yang sudah selesai otomatis dilewati, dan training CNN dilanjutkan dari epoch terakhir.
4. File akhir: `MyDrive/package-qc/submission.csv`.

Perkiraan waktu total di T4: ± 1–1,5 jam (unduh data ± 5 menit, audit + cache ± 10 menit, embedding ± 10 menit, ConvNeXt ± 30–40 menit).

---
# §0 · Persiapan Lingkungan
Library yang dipakai:

| Library | Dipakai untuk |
|---|---|
| `pandas`, `numpy` | Tabel & array |
| `Pillow` | Membuka gambar dari format apa pun (`.jpg` yang ternyata WEBP/PNG/AVIF) |
| `imagehash` | phash & dhash (sidik jari visual gambar) |
| `scipy` | *connected components* untuk menghitung gugus foto kembar, `median_filter` |
| `scikit-learn` | split stratified, StandardScaler, metrik F1 |
| `torch`, `torchvision` | Deep learning |
| `timm` | ConvNeXt, DINOv2 (dan EfficientNetV2 opsional) |
| `open_clip_torch` | CLIP ViT-B/16 bobot LAION-2B |

In [ ]:
%pip install -q open_clip_torch timm imagehash huggingface_hub
# dukungan AVIF untuk Pillow versi lama (opsional; bila gagal dipasang, pipeline tetap jalan)
%pip install -q pillow-avif-plugin

In [ ]:
import os, sys, json, math, time, random, hashlib, itertools, collections, warnings, shutil, zipfile
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, classification_report

try:
    import pillow_avif  # noqa: F401  (dukungan AVIF untuk versi Pillow lama)
except ImportError:
    pass

Image.MAX_IMAGE_PIXELS = None          # beberapa foto berukuran sangat besar
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 0.1 Konfigurasi
Semua path, saklar eksperimen, dan hyperparameter dikumpulkan di sini.

- **Data mentah & cache gambar** disimpan di disk lokal Colab (`/content`) karena jauh lebih cepat dibaca daripada Google Drive.
- **Hasil kerja** (`WORK`) disimpan di Google Drive supaya tidak hilang saat sesi putus.
- `USE_PSEUDO_OVERRIDE = False`: foto test yang identik dengan foto train **tidak** diberi label train secara langsung (lihat §9.4). Nyalakan hanya bila aturan lomba mengizinkan.

In [ ]:
CLASSES = ["Aman", "Rusak", "Lainnya"]
C2I = {c: i for i, c in enumerate(CLASSES)}
SEED = 42
HF_REPO = "Zhuhn/Package-QC-dataset"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                       # simpan hasil di Google Drive (disarankan di Colab)

LOCAL_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR  = LOCAL_DIR / "data"         # dataset mentah dari Hugging Face
CACHE_DIR = LOCAL_DIR / "cache_448"    # salinan gambar yang sudah diperkecil (sisi pendek 448 px)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/package-qc")
else:
    WORK = LOCAL_DIR / "work"

# ---- saklar tahap ----------------------------------------------------------
RUN = dict(
    mlp_base=True,        # §6  MLP di atas CLIP+DINOv2            (ensemble final)
    mlp_multicrop=True,   # §7  MLP di atas CLIP multi-crop + base  (ensemble final)
    convnext_r1=True,     # §8  ConvNeXt-tiny putaran 1             (ensemble final)
    convnext_r2=True,     # §8  ConvNeXt-tiny putaran 2             (ensemble final)
    # ---- eksperimen pembanding (§11), tidak masuk ensemble final ----
    zeroshot=False, logreg=False, knn=False, svm=False,
    vit_l14=False, effnet_focal=False, yolo=False,
)
USE_PSEUDO_OVERRIDE = False   # override 467-an foto test yang identik dengan train (cek aturan lomba!)
TWIN_AVERAGE = True           # rata-ratakan prediksi antar foto test yang kembar

# ---- hyperparameter --------------------------------------------------------
CACHE_SIDE = 448              # sisi pendek gambar cache (cukup untuk multi-crop 448 dan CNN 224)
EMB_BS = 64                   # batch ekstraksi embedding
MLP_CFG = dict(epochs=60, patience=12, lr=1e-3, wd=5e-2, bs=256, ls=0.1, seeds=(0, 1, 2))

CNN_COMMON = dict(model="convnext_tiny.fb_in22k_ft_in1k", bs=48, lr=1e-4, head_lr=1e-3, wd=0.05,
                  loss="ce", label_smoothing=0.1, drop_rate=0.0, drop_path=0.1, grad_clip=None,
                  tta=True, max_minutes=None)
CNN_CFGS = {
    # putaran 1: 192 px, stem + 2 stage dibekukan, augmentasi cukup agresif, sampler n^-0.75
    "convnext-r1": dict(CNN_COMMON, img=192, epochs=5, freeze_stages=2, sampler_power=0.75,
                        train_aug="r1", eval_tf="crop"),
    # putaran 2: 224 px, semua layer dilatih, augmentasi ringan (tanpa noise/rotasi), letterbox, sampler n^-0.5
    "convnext-r2": dict(CNN_COMMON, img=224, epochs=7, freeze_stages=0, sampler_power=0.5,
                        train_aug="r2", eval_tf="letterbox"),
}
# komponen ensemble final dan bobot awalnya (bobot & bias dicari ulang di §9)
ENSEMBLE_RUNS = ["emb-r2-mlp", "emb-heads-mlp", "convnext-r1", "convnext-r2"]

# ---- perangkat --------------------------------------------------------------
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEV.type == "cuda"
# bf16 hanya di GPU Ampere ke atas (L4/A100); T4 memakai fp16 + GradScaler
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
NUM_WORKERS = min(8, os.cpu_count() or 2)

SPLIT_DIR = WORK / "split"
RUNS = WORK / "runs"
EDA = RUNS / "eda"
EMB = RUNS / "emb"
for d in (WORK, SPLIT_DIR, RUNS, EDA, EMB):
    d.mkdir(parents=True, exist_ok=True)
print("WORK  :", WORK)
print("DATA  :", DATA_DIR)
print("CACHE :", CACHE_DIR)
print("device:", DEV, "| AMP:", USE_AMP, AMP_DTYPE if USE_AMP else "", "| workers:", NUM_WORKERS)

## 0.2 Fungsi bantu
- `seed_everything`: membuat hasil bisa diulang.
- `macro_f1`: versi cepat (numpy) dari `f1_score(average="macro")`, dipakai ribuan kali saat mencari bobot ensemble.
- `metrics`: menghitung Macro-F1 di **val penuh** dan **val bersih** sekaligus (lihat §3.5).
- `write_contract`: setiap model menulis tiga file dengan format yang sama persis (*file contract*), sehingga ensemble cukup membaca CSV tanpa peduli model apa yang menghasilkannya:

| File | Kolom |
|---|---|
| `val_probs.csv` | `path,label,p_Aman,p_Rusak,p_Lainnya` |
| `test_probs.csv` | `filename,p_Aman,p_Rusak,p_Lainnya` |
| `submission.csv` | `filename,label` |

In [ ]:
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def macro_f1(y, pred, k=3):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1)
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return f1.mean()


def metrics(y, P, cmask=None):
    pred = P.argmax(1)
    out = {"macro_f1": float(f1_score(y, pred, average="macro")),
           "per_class_f1": dict(zip(CLASSES, np.round(f1_score(y, pred, average=None, labels=[0, 1, 2]), 4).tolist())),
           "cm": confusion_matrix(y, pred, labels=[0, 1, 2]).tolist()}
    if cmask is not None:
        out["macro_f1_clean"] = float(f1_score(y[cmask], pred[cmask], average="macro"))
        out["per_class_f1_clean"] = dict(zip(CLASSES, np.round(
            f1_score(y[cmask], pred[cmask], average=None, labels=[0, 1, 2]), 4).tolist()))
        out["cm_clean"] = confusion_matrix(y[cmask], pred[cmask], labels=[0, 1, 2]).tolist()
    return out


def run_done(name):
    d = RUNS / name
    return (d / "val_probs.csv").exists() and (d / "test_probs.csv").exists()


def write_contract(name, pva, pte, extra=None):
    # pva: (n_val,3) urut sesuai split/val.csv ; pte: (n_test,3) urut sesuai TEST_FILES
    d = RUNS / name
    d.mkdir(parents=True, exist_ok=True)
    assert pva.shape == (len(VAL), 3) and pte.shape == (len(TEST_FILES), 3)
    pd.DataFrame({"path": VAL.path, "label": VAL.label, "p_Aman": pva[:, 0], "p_Rusak": pva[:, 1],
                  "p_Lainnya": pva[:, 2]}).to_csv(d / "val_probs.csv", index=False)
    pd.DataFrame({"filename": TEST_FILES, "p_Aman": pte[:, 0], "p_Rusak": pte[:, 1],
                  "p_Lainnya": pte[:, 2]}).to_csv(d / "test_probs.csv", index=False)
    sub = pd.DataFrame({"filename": TEST_FILES, "label": [CLASSES[i] for i in pte.argmax(1)]})
    sub = SAMPLE[["filename"]].merge(sub, on="filename", how="left")
    assert sub.label.notna().all() and len(sub) == len(SAMPLE)
    sub.to_csv(d / "submission.csv", index=False)
    m = metrics(Y_VAL, pva, CLEAN_MASK)
    if extra:
        m.update(extra)
    json.dump(m, open(d / "metrics.json", "w"), indent=1)
    print(f"[{name}] val Macro-F1 = {m['macro_f1']:.4f} | val bersih = {m['macro_f1_clean']:.4f} | "
          f"per kelas (bersih) = {m['per_class_f1_clean']}")
    print(f"[{name}] distribusi prediksi test:", sub.label.value_counts().to_dict())
    return m


def show_metrics(name, m):
    print(f"== {name}: Macro-F1 val penuh {m['macro_f1']:.4f} | val bersih {m['macro_f1_clean']:.4f}")
    print("   per kelas (penuh) :", m["per_class_f1"])
    print("   per kelas (bersih):", m["per_class_f1_clean"])
    print("   confusion (baris = label asli, kolom = prediksi; urut Aman, Rusak, Lainnya):")
    print(np.array(m["cm"]))


seed_everything()

---
# §1 · Data
## 1.1 Unduh dataset dari Hugging Face
Dataset (± 3,3 GB) diunduh sekali ke `/content/data`. Bila folder sudah berisi `Train_Set` dan `Test_Set`, unduhan dilewati. Bila dataset berupa arsip `.zip`, arsip diekstrak otomatis.

> Repo dataset publik, jadi tidak perlu token. Bila unduhan terkena batas laju (*HTTP 429*), tambahkan token Hugging Face sebagai *Colab secret* bernama `HF_TOKEN` (ikon kunci di panel kiri); `huggingface_hub` akan memakainya otomatis.

Struktur yang diharapkan:
```
data/
├── Train_Set/{Aman, Rusak, Lainnya}/   (8.800 / 3.001 / 423 foto)
├── Test_Set/                           (3.057 foto)
└── sample_submission.csv
```

In [ ]:
def find_data_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for root, dirs, _ in os.walk(base):
        if "Train_Set" in dirs and "Test_Set" in dirs:
            return Path(root)
    return None


DATA_ROOT = find_data_root(DATA_DIR)
if DATA_ROOT is None:
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=str(DATA_DIR), max_workers=16)
    for z in sorted(DATA_DIR.rglob("*.zip")):
        print("ekstrak", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.parent)
    DATA_ROOT = find_data_root(DATA_DIR)
    print(f"unduh selesai dalam {time.time() - t0:.0f} s")
assert DATA_ROOT is not None, f"Folder Train_Set/Test_Set tidak ditemukan di {DATA_DIR}"

_ss = [p for p in [DATA_ROOT / "sample_submission.csv"] + sorted(DATA_DIR.rglob("sample_submission.csv")) if p.exists()]
assert _ss, "sample_submission.csv tidak ditemukan"
SAMPLE = pd.read_csv(_ss[0])

TEST_DIR = DATA_ROOT / "Test_Set"
TEST_FILES = sorted(f for f in os.listdir(TEST_DIR) if not f.startswith("."))
print("DATA_ROOT:", DATA_ROOT)
for c in CLASSES:
    print(f"  Train_Set/{c:8s}: {len([f for f in os.listdir(DATA_ROOT / 'Train_Set' / c) if not f.startswith('.')]):5d} foto")
print(f"  Test_Set        : {len(TEST_FILES):5d} foto")
print(f"  sample_submission: {len(SAMPLE)} baris; kolom {list(SAMPLE.columns)}")
assert set(SAMPLE.filename) == set(TEST_FILES), "isi Test_Set tidak sama dengan sample_submission"

## 1.2 Melihat contoh foto
Lima foto acak per kelas. Perhatikan: sebagian foto *Aman* diputar (ada sudut hitam) atau diberi *noise* bintik. Ini artefak augmentasi dari pembuat dataset dan akan diukur di §3.4.

In [ ]:
def show_grid(paths, titles, ncol=5, size=2.6):
    nrow = math.ceil(len(paths) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, p, t in zip(np.ravel(axes), paths, titles):
        try:
            ax.imshow(Image.open(p).convert("RGB"))
        except Exception as e:
            ax.text(0.5, 0.5, f"gagal dibuka\n{type(e).__name__}", ha="center")
        ax.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()


rng = random.Random(SEED)
_paths, _titles = [], []
for c in CLASSES:
    fs = sorted(os.listdir(DATA_ROOT / "Train_Set" / c))
    for f in rng.sample(fs, min(5, len(fs))):
        _paths.append(DATA_ROOT / "Train_Set" / c / f); _titles.append(c)
show_grid(_paths, _titles)

---
# §2 · Tahap 1: Membagi Data (Split)
Test_Set resmi tidak berlabel, jadi kita memisahkan **15 %** data berlabel sebagai *ujian internal* (val).

| | |
|---|---|
| **Stratified** | proporsi kelas dijaga sama di train dan val (tanpa ini, 423 foto *Lainnya* bisa menumpuk di satu sisi) |
| **seed 42** | pembagian bisa diulang persis; semua model memakai split yang sama supaya bisa dibandingkan |
| **Ukuran val** | `int(12.224 × 0,15)` = **1.833** foto (Aman 1.320 · Rusak 450 · Lainnya 63). `test_size=0.15` akan membulatkan ke 1.834, jadi dipakai jumlah bulat. |
| **Path relatif** | kolom `path` disimpan relatif terhadap folder data (mis. `Train_Set/Aman/Aman_00001.jpg`), sehingga CSV bisa dipakai di komputer mana pun |

Val **tidak pernah** dipakai untuk melatih bobot model; hanya untuk memilih epoch/hyperparameter dan mencari bobot ensemble.

In [ ]:
if (SPLIT_DIR / "train.csv").exists() and (SPLIT_DIR / "val.csv").exists():
    TRAIN = pd.read_csv(SPLIT_DIR / "train.csv"); VAL = pd.read_csv(SPLIT_DIR / "val.csv")
    print("split dimuat dari", SPLIT_DIR)
else:
    rows = []
    for label in CLASSES:
        d = DATA_ROOT / "Train_Set" / label
        for fn in sorted(os.listdir(d)):            # urutan deterministik
            if fn.startswith("."):
                continue                            # lewati file tersembunyi
            rows.append((f"Train_Set/{label}/{fn}", label))
    df = pd.DataFrame(rows, columns=["path", "label"])
    n_val = int(len(df) * 0.15)
    TRAIN, VAL = train_test_split(df, test_size=n_val, stratify=df["label"], random_state=SEED)
    TRAIN.to_csv(SPLIT_DIR / "train.csv", index=False); VAL.to_csv(SPLIT_DIR / "val.csv", index=False)
    TRAIN = pd.read_csv(SPLIT_DIR / "train.csv"); VAL = pd.read_csv(SPLIT_DIR / "val.csv")
    print("split baru ditulis ke", SPLIT_DIR)

Y_TR = TRAIN.label.map(C2I).values
Y_VAL = VAL.label.map(C2I).values
print("train:", len(TRAIN), TRAIN.label.value_counts().to_dict())
print("val  :", len(VAL), VAL.label.value_counts().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for a, (nm, d) in zip(ax, [("train", TRAIN), ("val", VAL)]):
    vc = d.label.value_counts().reindex(CLASSES)
    a.bar(CLASSES, vc.values, color=["#4C72B0", "#C44E52", "#55A868"])
    for i, v in enumerate(vc.values):
        a.text(i, v, f"{v}\n({v / len(d):.1%})", ha="center", va="bottom", fontsize=9)
    a.set_title(f"Distribusi kelas · {nm}"); a.set_ylim(0, vc.max() * 1.25)
plt.tight_layout(); plt.show()

### Kenapa Macro-F1, bukan akurasi?
Kelas *Lainnya* hanya ± 3,5 % data, tetapi bobotnya **sepertiga** skor. Model yang selalu menjawab "Aman" mendapat akurasi 72 %, tetapi Macro-F1 hanya ± 0,28. Karena itu seluruh pipeline dirancang untuk **menjaga kelas kecil**: sampler berbobot, bobot kelas, bias logit per kelas, dan pemilihan model berdasarkan Macro-F1.

---
# §3 · Tahap 2: Audit Data (EDA)
Sebelum melatih model apa pun, seluruh ± 15 ribu foto (train + test) diperiksa **tanpa diubah**. Audit ini menentukan cara membaca gambar, augmentasi, cara mengukur val, dan pasca-proses.

## 3.1 Memindai semua foto: md5, phash, dhash, artefak augmentasi (+ membuat cache)

| Teknik | Apa itu | Gunanya di sini |
|---|---|---|
| **md5** | sidik jari 32 karakter dari seluruh *byte* file | menemukan file **identik persis** |
| **phash** | gambar → grayscale 32×32 → DCT → blok frekuensi rendah 8×8 → bandingkan dengan median → 64 bit | menemukan *near-duplicate* (beda resolusi, kompresi, sedikit noise) |
| **dhash** | grayscale 9×8, tiap piksel dibandingkan tetangga kanannya → 64 bit | melengkapi phash (arah gradien) |
| **Jarak Hamming** | banyaknya bit berbeda antara dua hash (0–64) | **jarak ≤ 6 dianggap kembar** |
| **black_corner** | % piksel hitam di empat sudut (padding hasil rotasi) | mendeteksi foto yang diputar |
| **saltpepper** | % piksel yang berbeda > 60 dari median 3×3-nya | mendeteksi noise bintik |
| **colorfulness** | selisih rata-rata antar kanal warna (≈ 0 = grayscale) | mendeteksi foto grayscale |

Dalam satu kali baca, setiap foto juga disimpan sebagai **cache JPEG** dengan sisi pendek 448 px. Foto asli berukuran sampai ± 4000 px; men-*decode*-nya berulang kali di setiap epoch membuat DataLoader menjadi penghambat. Semua model setelah ini membaca versi cache.

Catatan pembacaan gambar:
- `Image.open(...).convert("RGB")` membaca **isi** file, bukan ekstensinya. Ribuan file `.jpg` di dataset ini sebenarnya WEBP/PNG/AVIF; `cv2.imread` bisa gagal diam-diam pada file seperti itu.
- `img.draft("RGB", ...)`: untuk JPEG besar, decoder langsung menghasilkan versi 1/2, 1/4, atau 1/8 ukuran sehingga jauh lebih cepat.

In [ ]:
from scipy.ndimage import median_filter


def cache_path(rel):
    # nama asli + ".jpg" supaya file dengan nama sama tetapi ekstensi berbeda tidak bertabrakan
    return CACHE_DIR / (rel + ".jpg")


def load_rgb(path, draft_size=None):
    img = Image.open(path)
    if draft_size and img.format == "JPEG":
        img.draft("RGB", (draft_size, draft_size))
    return img.convert("RGB")


def aug_scores(rgb):
    g = np.asarray(rgb.convert("L").resize((128, 128)), dtype=np.int16)
    c = 12
    corners = np.concatenate([g[:c, :c].ravel(), g[:c, -c:].ravel(), g[-c:, :c].ravel(), g[-c:, -c:].ravel()])
    black = float((corners <= 3).mean())
    sp = float((np.abs(g - median_filter(g, size=3)) > 60).mean())
    s = np.asarray(rgb.resize((64, 64)), dtype=np.int16)
    sat = float(np.abs(s[..., 0] - s[..., 1]).mean() + np.abs(s[..., 1] - s[..., 2]).mean())
    return black, sp, sat


def scan_one(job):
    split, label, rel, need_meta = job
    src = DATA_ROOT / rel
    rec = dict(split=split, label=label, path=rel, filename=os.path.basename(rel),
               ext=os.path.splitext(rel)[1].lower(), bytes=0, md5="", fmt="", mode="", w=0, h=0,
               has_alpha=0, phash="", dhash="", black_corner=np.nan, saltpepper=np.nan,
               colorfulness=np.nan, error="")
    try:
        if need_meta:
            raw = open(src, "rb").read()
            rec["bytes"] = len(raw)
            rec["md5"] = hashlib.md5(raw).hexdigest()
        im = Image.open(src)
        rec["fmt"] = im.format or ""; rec["mode"] = im.mode; rec["w"], rec["h"] = im.size
        rec["has_alpha"] = int(im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info))
        if im.format == "JPEG":
            im.draft("RGB", (CACHE_SIDE * 2, CACHE_SIDE * 2))
        im.load()                                   # paksa decode penuh -> file terpotong ketahuan
        rgb = im.convert("RGB")
        if need_meta:
            rec["phash"] = str(imagehash.phash(rgb)); rec["dhash"] = str(imagehash.dhash(rgb))
            rec["black_corner"], rec["saltpepper"], rec["colorfulness"] = aug_scores(rgb)
        dst = cache_path(rel)
        if not dst.exists():
            w, h = rgb.size; s = CACHE_SIDE / min(w, h)
            if s < 1:                               # hanya mengecilkan, tidak memperbesar
                rgb = rgb.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)
            dst.parent.mkdir(parents=True, exist_ok=True)
            rgb.save(dst, "JPEG", quality=93)
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
    return rec


META_CSV = EDA / "meta.csv"
jobs = [("train", lab, p) for p, lab in pd.concat([TRAIN, VAL])[["path", "label"]].itertuples(index=False)]
jobs += [("test", "", f"Test_Set/{f}") for f in TEST_FILES]
have_meta = META_CSV.exists()
missing_cache = [j for j in jobs if not cache_path(j[2]).exists()]
if have_meta and not missing_cache:
    print("meta.csv dan cache sudah lengkap, dilewati")
else:
    todo = jobs if not have_meta else missing_cache
    todo = [(s, l, p, not have_meta) for s, l, p in todo]
    print(f"memproses {len(todo)} foto ({'meta + cache' if not have_meta else 'cache saja'}) dengan {os.cpu_count()} proses")
    t0 = time.time(); recs = []
    with Pool(os.cpu_count()) as pool:
        for i, rec in enumerate(pool.imap_unordered(scan_one, todo, chunksize=16)):
            recs.append(rec)
            if i % 2000 == 0:
                print(f"  {i}/{len(todo)}  {time.time() - t0:.0f}s", flush=True)
    if not have_meta:
        pd.DataFrame(recs).sort_values(["split", "path"]).to_csv(META_CSV, index=False)
    print(f"selesai {time.time() - t0:.0f}s")

META = pd.read_csv(META_CSV, dtype={"md5": str, "phash": str, "dhash": str}, keep_default_na=False,
                   na_values={"black_corner": [""], "saltpepper": [""], "colorfulness": [""]})
META["fold"] = META.path.map({**{p: "train" for p in TRAIN.path}, **{p: "val" for p in VAL.path}}).fillna("test")
print("total foto:", len(META), "| gagal dibaca:", (META.error != "").sum())
if (META.error != "").any():
    display(META[META.error != ""][["path", "error"]])

### Format sebenarnya vs ekstensi file

In [ ]:
META["ext_vs_fmt"] = META.ext + " → " + META.fmt
display(pd.crosstab(META.ext_vs_fmt, META.split))
lie = META[(META.ext.isin([".jpg", ".jpeg"])) & (META.fmt != "JPEG")]
print(f"file berekstensi .jpg yang sebenarnya bukan JPEG: train {(lie.split == 'train').sum()} · test {(lie.split == 'test').sum()}")
print("foto bertransparansi (alpha):", META.groupby("split").has_alpha.sum().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s, col in [("train", "#4C72B0"), ("test", "#DD8452")]:
    d = META[META.split == s]
    ax[0].scatter(d.w, d.h, s=3, alpha=0.3, label=s, color=col)
    ax[1].hist(np.log10(np.maximum(d.bytes, 1)), bins=60, alpha=0.5, label=s, color=col)
ax[0].set_xlabel("lebar (px)"); ax[0].set_ylabel("tinggi (px)"); ax[0].set_title("Ukuran gambar"); ax[0].legend()
ax[1].set_xlabel("log10(ukuran file, byte)"); ax[1].set_title("Ukuran file"); ax[1].legend()
plt.tight_layout(); plt.show()
print("median sisi terpendek per kelas:",
      META.assign(s=META[["w", "h"]].min(axis=1)).groupby(META.label.replace("", "(test)")).s.median().to_dict())
print("persen foto test tidak persegi:", f"{(META[META.split == 'test'].w != META[META.split == 'test'].h).mean():.1%}")

## 3.2 Mencari duplikat dan label konflik
Fungsi `hamming` menghitung jarak Hamming antara semua pasangan hash sekaligus: hash 64-bit disimpan sebagai `uint64`, di-XOR, lalu jumlah bit 1 dihitung. Pemrosesan dipotong per blok agar matriks jarak tidak memenuhi memori.

In [ ]:
_POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def hash_to_u64(series):
    return np.array([int(h, 16) if isinstance(h, str) and h else 0 for h in series], dtype=np.uint64)


def hamming(A, B):
    # A: (n,) uint64, B: (m,) uint64 -> (n, m) jarak Hamming
    x = A[:, None] ^ B[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return _POP8[x.view(np.uint8)].reshape(*x.shape, 8).sum(-1).astype(np.int16)


def nn_hamming(A, B, chunk=512):
    # untuk tiap elemen A: jarak & indeks tetangga terdekat di B
    d_min, i_min = [], []
    for s in range(0, len(A), chunk):
        d = hamming(A[s:s + chunk], B)
        i_min.append(d.argmin(1)); d_min.append(d.min(1))
    return np.concatenate(d_min), np.concatenate(i_min)


ok = META.error == ""
TR_ALL = META[(META.split == "train") & ok].reset_index(drop=True)     # 12.224 foto berlabel (train + val)
TE = META[(META.split == "test") & ok].reset_index(drop=True)
P_tr, D_tr = hash_to_u64(TR_ALL.phash), hash_to_u64(TR_ALL.dhash)
P_te, D_te = hash_to_u64(TE.phash), hash_to_u64(TE.dhash)

# ---- B1. duplikat byte-identik di dalam data berlabel, dan label konflik ----
g = TR_ALL.groupby("md5")
dup_groups = [d for _, d in g if len(d) > 1]
conflict = [d for d in dup_groups if d.label.nunique() > 1]
leak = sum(1 for d in dup_groups if d.fold.nunique() > 1)
print("== B1. md5 kembar di data berlabel")
print(f"  grup: {len(dup_groups)} | file terlibat: {sum(map(len, dup_groups))} | "
      f"file redundan: {sum(map(len, dup_groups)) - len(dup_groups)}")
print(f"  grup LABEL KONFLIK: {len(conflict)}", collections.Counter(tuple(sorted(set(d.label))) for d in conflict))
print(f"  grup kembar yang terbelah antara train dan val: {leak}")

# ---- B2. kebocoran test <-> train (md5 identik) ----
md5_lab = TR_ALL.groupby("md5").label.agg(lambda s: tuple(sorted(set(s))))
hit = TE[TE.md5.isin(md5_lab.index)]
print("\n== B2. foto test yang byte-identik dengan foto train:", len(hit),
      dict(collections.Counter(md5_lab[hit.md5].values)))
tg = TE.groupby("md5").size()
print("  grup md5 kembar di dalam test:", (tg > 1).sum())

# ---- B3. tetangga terdekat phash test -> train ----
nn_d, nn_i = nn_hamming(P_te, P_tr)
TE["nn_phash_d"] = nn_d; TE["nn_train_idx"] = nn_i
print("\n== B3. distribusi jarak phash test -> train terdekat")
for thr in (0, 2, 4, 6, 8, 10):
    print(f"  jarak <= {thr:2d}: {(nn_d <= thr).sum():5d} foto test")
print(f"  => {(nn_d <= 6).sum()} foto test ({(nn_d <= 6).mean():.1%}) punya kembaran di train; "
      f"{(nn_d > 6).sum()} benar-benar baru")

plt.figure(figsize=(7, 3))
plt.hist(np.minimum(nn_d, 30), bins=31, color="#4C72B0")
plt.axvline(6.5, color="red", ls="--", label="ambang kembar (≤ 6)")
plt.xlabel("jarak phash ke foto train terdekat (dipotong di 30)"); plt.ylabel("jumlah foto test"); plt.legend()
plt.title("Seberapa banyak foto test yang mirip foto train?"); plt.tight_layout(); plt.show()

### Pasangan test–train dan *pseudo-label*
Semua pasangan test–train dengan jarak phash ≤ 6 disimpan. Dari situ dibuat dua daftar:
- **strict**: tetangga dekat (≤ 4) dan **semua** tetangga ≤ 6 berlabel sama.
- **strong** (dipakai untuk override opsional di §9): md5 identik **atau** (phash 0 dan dhash ≤ 4), dan tidak ada tetangga ≤ 6 yang berlabel lain.

In [ ]:
THR = 6
rows = []
for s in range(0, len(TE), 256):
    d = hamming(P_te[s:s + 256], P_tr)
    for ii, jj in zip(*np.where(d <= THR)):
        i = s + ii
        rows.append(dict(test_filename=TE.filename[i], train_path=TR_ALL.path[jj], label=TR_ALL.label[jj],
                         hash_distance=int(d[ii, jj]), dhash_distance=int(hamming(D_te[i:i + 1], D_tr[jj:jj + 1])[0, 0]),
                         exact_md5=int(TE.md5[i] == TR_ALL.md5[jj]), train_fold=TR_ALL.fold[jj]))
MT = pd.DataFrame(rows, columns=["test_filename", "train_path", "label", "hash_distance", "dhash_distance",
                                 "exact_md5", "train_fold"]).sort_values(["test_filename", "hash_distance"])
MT.to_csv(EDA / "test_train_matches.csv", index=False)
print(f"test_train_matches.csv: {len(MT)} pasangan, {MT.test_filename.nunique()} foto test unik")

agg = MT.groupby("test_filename").agg(labels=("label", lambda s: tuple(sorted(set(s)))), mind=("hash_distance", "min"))
print("konsistensi label tetangga per foto test:", dict(collections.Counter(agg.labels)))
strict = agg[(agg.mind <= 4) & (agg.labels.apply(len) == 1)]
strict = strict.assign(label=strict.labels.str[0])[["label", "mind"]]
strict.to_csv(EDA / "test_pseudo_labels_strict.csv")
print("pseudo-label strict:", len(strict), strict.label.value_counts().to_dict())

strong_rows = []
for fn, d in MT.groupby("test_filename"):
    labs = set(d.label)
    if len(labs) != 1:
        continue                                   # ada tetangga <= 6 berlabel beda -> tidak aman
    kuat = ((d.exact_md5 == 1) | ((d.hash_distance == 0) & (d.dhash_distance <= 4))).any()
    if kuat:
        strong_rows.append((fn, labs.pop()))
PSEUDO = pd.DataFrame(strong_rows, columns=["test_filename", "label"])
PSEUDO.to_csv(EDA / "test_pseudo_labels_strong.csv", index=False)
print("pseudo-label strong:", len(PSEUDO), PSEUDO.label.value_counts().to_dict())

### Near-duplicate di dalam data berlabel dan *connected components*
Jika foto A mirip B dan B mirip C, ketiganya dianggap satu gugus walau A dan C tidak langsung mirip. `scipy.sparse.csgraph.connected_components` menghitung gugus itu dari daftar pasangan. Dari sini terlihat berapa foto yang benar-benar unik dan berapa pasangan mirip yang labelnya berbeda (*noise label*).

In [ ]:
import scipy.sparse as sps
from scipy.sparse.csgraph import connected_components

pairs = []
for s in range(0, len(TR_ALL), 512):
    d = hamming(P_tr[s:s + 512], P_tr)
    for ii, jj in zip(*np.where(d <= THR)):
        if jj > s + ii:
            pairs.append((s + ii, jj, int(d[ii, jj])))
PP = pd.DataFrame(pairs, columns=["i", "j", "d"])
PP["li"] = TR_ALL.label.values[PP.i]; PP["lj"] = TR_ALL.label.values[PP.j]
PP["same_md5"] = TR_ALL.md5.values[PP.i] == TR_ALL.md5.values[PP.j]
PP["fi"] = TR_ALL.fold.values[PP.i]; PP["fj"] = TR_ALL.fold.values[PP.j]
conf = PP[PP.li != PP.lj]
print(f"pasangan phash <= {THR}: {len(PP)} (identik md5: {PP.same_md5.sum()}, near-dup: {(~PP.same_md5).sum()})")
print("pasangan LINTAS LABEL:", len(conf))
if len(conf):
    display(pd.crosstab(conf.li, conf.lj))
conf.assign(fi_name=TR_ALL.path.values[conf.i], fj_name=TR_ALL.path.values[conf.j]) \
    .to_csv(EDA / "train_cross_label_pairs.csv", index=False)
print("pasangan near-dup yang terbelah train/val:", (PP.fi != PP.fj).sum())

A = sps.coo_matrix((np.ones(len(PP)), (PP.i, PP.j)), shape=(len(TR_ALL), len(TR_ALL)))
ncomp, lab = connected_components(A, directed=False)
sizes = pd.Series(lab).value_counts()
print(f"foto berlabel unik setelah menggabungkan near-dup: {ncomp} gugus dari {len(TR_ALL)} foto "
      f"({1 - ncomp / len(TR_ALL):.1%} redundan); gugus terbesar: {sizes.head(5).tolist()}")

## 3.3 Mengukur artefak augmentasi (jalan pintas / *shortcut*)
Bila rotasi dan noise lebih sering muncul pada satu kelas, model bisa belajar "ada noise → Aman" alih-alih melihat kondisi paket. Kalau artefak ini hampir tidak ada di test, augmentasi noise/rotasi berat saat training justru **memperkuat jalan pintas**. Karena itu ConvNeXt putaran 2 (§8) sengaja membuangnya.

In [ ]:
grp = META.assign(group=np.where(META.split == "test", "Test", "Train " + META.label))
summary = grp.groupby("group").agg(
    n=("path", "size"),
    rotasi_sudut_hitam=("black_corner", lambda s: (s > 0.5).mean()),
    noise_saltpepper=("saltpepper", lambda s: (s > 0.05).mean()),
    grayscale=("colorfulness", lambda s: (s < 2).mean()),
)
display(summary.style.format({"rotasi_sudut_hitam": "{:.1%}", "noise_saltpepper": "{:.1%}", "grayscale": "{:.1%}"}))
summary[["rotasi_sudut_hitam", "noise_saltpepper", "grayscale"]].plot.bar(figsize=(8, 3), rot=0)
plt.ylabel("proporsi foto"); plt.title("Artefak augmentasi per kelompok"); plt.tight_layout(); plt.show()

## 3.4 *Val bersih*
Sebagian foto val punya kembaran (md5 sama atau phash ≤ 6) di **train**. Model yang sekadar menghafal akan terlihat bagus di val, padahal di test sebagian besar foto benar-benar baru.

Solusinya: setiap model dilaporkan dengan **dua angka**:
- **val penuh**: seluruh 1.833 foto val.
- **val bersih**: hanya foto val yang **tidak** punya kembaran di train. Angka ini lebih mendekati skor sebenarnya di test, dan dipakai untuk memilih bobot/bias ensemble.

In [ ]:
trf = META[(META.fold == "train") & ok]
P_trf = hash_to_u64(trf.phash)
md5_trf = set(trf.md5)
vmeta = META.set_index("path").loc[VAL.path]
d_val, _ = nn_hamming(hash_to_u64(vmeta.phash), P_trf)
CLEAN_MASK = (~vmeta.md5.isin(md5_trf).values) & (d_val > THR) & (vmeta.error.values == "")
pd.DataFrame({"path": VAL.path, "clean": CLEAN_MASK}).to_csv(EDA / "val_clean.csv", index=False)
print(f"val total {len(VAL)} | val bersih {CLEAN_MASK.sum()} "
      f"(Aman/Rusak/Lainnya = {np.bincount(Y_VAL[CLEAN_MASK], minlength=3).tolist()}) | "
      f"punya kembaran di train: {(~CLEAN_MASK).sum()} ({(~CLEAN_MASK).mean():.1%})")

### Grup foto test yang kembar
Foto test yang kembar seharusnya mendapat jawaban yang sama. Foto dikelompokkan dengan *union-find*: dua foto test masuk satu grup jika md5 sama **atau** jarak phash ≤ 6. Di §9, prediksi di dalam satu grup dirata-ratakan.

In [ ]:
def union_find_groups(n, edges):
    par = list(range(n))

    def find(i):
        while par[i] != i:
            par[i] = par[par[i]]; i = par[i]
        return i

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb:
            par[rb] = ra
    return np.array([find(i) for i in range(n)])


te_all = META[META.split == "test"].set_index("filename").loc[TEST_FILES]
P_all = hash_to_u64(te_all.phash)
valid = (te_all.error.values == "")
edges = []
for s in range(0, len(P_all), 512):
    d = hamming(P_all[s:s + 512], P_all)
    for ii, jj in zip(*np.where(d <= THR)):
        i = s + ii
        if jj > i and valid[i] and valid[jj]:
            edges.append((i, jj))
by_md5 = collections.defaultdict(list)
for i, m in enumerate(te_all.md5.values):
    if m:
        by_md5[m].append(i)
for g_ in by_md5.values():
    edges += [(g_[0], k) for k in g_[1:]]
TEST_GRP = union_find_groups(len(TEST_FILES), edges)
pd.DataFrame({"filename": TEST_FILES, "grp": TEST_GRP}).to_csv(EDA / "test_groups.csv", index=False)
print("grup foto test kembar (ukuran > 1):", (pd.Series(TEST_GRP).value_counts() > 1).sum())

## 3.5 Ringkasan keputusan dari audit

| Temuan | Keputusan di tahap berikutnya |
|---|---|
| Ekstensi bohong (WEBP/PNG/AVIF berekstensi `.jpg`) | Selalu `PIL.Image.open(...).convert("RGB")` |
| Banyak foto test tidak persegi, objek sering kecil | Resize **letterbox** (padding), bukan center-crop |
| Val terinflasi foto kembar | Laporkan Macro-F1 val penuh **dan** val bersih; pilih bobot ensemble di val bersih |
| Noise/rotasi = jalan pintas | Putaran 2 tanpa augmentasi noise/rotasi/grayscale/erasing |
| Test punya foto kembar sendiri | Rata-ratakan prediksi antar kembar sebelum argmax |
| Foto test identik dengan train, label konsisten | Override opsional (cek aturan lomba) |
| *Lainnya* sangat sedikit | Sampler berbobot, bobot kelas, bias logit per kelas |

---
# §4 · Tahap 3: Membaca dan Menyiapkan Gambar
## Letterbox vs center-crop
| | |
|---|---|
| **Letterbox** | perkecil gambar sampai sisi **terpanjang** = ukuran target, lalu taruh di tengah kanvas persegi abu-abu. Rasio aspek tidak berubah dan tidak ada bagian yang terpotong. |
| **Center-crop** | memotong tepi; bisa membuang bagian paket yang penyok/sobek. |
| **Squash** | `Resize((s, s))` mengubah rasio sehingga objek tampak gepeng. |

Warna padding (124, 116, 104) mendekati rata-rata warna ImageNet sehingga padding "netral" bagi model. `Normalize(MEAN, STD)` wajib karena model pretrained dilatih dengan normalisasi itu.

`CsvDataset` tahan file rusak: saat training, file yang gagal dibaca diganti sampel acak lain; saat evaluasi, file rusak menghasilkan tensor nol dengan penanda `ok=0` sehingga probabilitasnya diganti seragam 1/3.

In [ ]:
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class Letterbox:
    # Resize sisi TERPANJANG = size, lalu padding ke size x size (seluruh gambar tetap terlihat)
    def __init__(self, size, fill=(124, 116, 104)):
        self.size, self.fill = size, fill

    def __call__(self, img):
        w, h = img.size; s = self.size / max(w, h)
        nw, nh = max(1, round(w * s)), max(1, round(h * s))
        img = img.resize((nw, nh), Image.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(img, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas


def src_path(rel):
    # pakai cache bila ada, kalau tidak baca file asli
    c = cache_path(rel)
    return c if c.exists() else DATA_ROOT / rel


VAL_REL = VAL.path.tolist()
TRAIN_REL = TRAIN.path.tolist()
TEST_REL = [f"Test_Set/{f}" for f in TEST_FILES]
SPLITS = {"train": TRAIN_REL, "val": VAL_REL, "test": TEST_REL}


class CsvDataset(Dataset):
    def __init__(self, rels, labels, tf, size, train=False):
        self.rels, self.labels, self.tf, self.size, self.train = list(rels), labels, tf, size, train

    def __len__(self):
        return len(self.rels)

    def __getitem__(self, i):
        y = int(self.labels[i]) if self.labels is not None else -1
        try:
            img = load_rgb(src_path(self.rels[i]), draft_size=self.size * 2)
            return self.tf(img), y, 1
        except Exception as e:
            print(f"CORRUPT/SKIP {self.rels[i]}: {e}")
            if self.train:
                return self[random.randrange(len(self))]
            return torch.zeros(3, self.size, self.size), y, 0


# contoh letterbox vs center-crop
_ex = META[(META.split == "test") & (META.w > 1.6 * META.h)].head(1).path.tolist() or [TEST_REL[0]]
_img = load_rgb(src_path(_ex[0]))
fig, ax = plt.subplots(1, 3, figsize=(10, 3.3))
for a, (t, im) in zip(ax, [("asli", _img), ("letterbox 224 (dipakai)", Letterbox(224)(_img)),
                           ("center-crop 224 (dihindari)", T.CenterCrop(224)(T.Resize(224)(_img)))]):
    a.imshow(im); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

---
# §5 · Tahap 4: Embedding dari *Foundation Model*
Alih-alih melatih jaringan dari awal, kita meminjam model besar yang sudah "melihat" ratusan juta gambar. Model itu dibekukan (*frozen*) dan hanya dipakai untuk mengubah tiap foto menjadi vektor angka (**embedding**). Masalah "mengerti gambar" berubah menjadi masalah tabel biasa (10.391 baris × 896 kolom) yang bisa ditangani classifier kecil dalam hitungan detik.

| Model | Apa itu | Keluaran | Gunanya di sini |
|---|---|---|---|
| **CLIP ViT-B/16** (OpenCLIP, `laion2b_s34b_b88k`) | dilatih *contrastive* pada ± 2 miliar pasangan gambar–teks | 512-d | kuat di **semantik/konteks**: kardus pengiriman vs kemasan makanan vs kantong plastik → kelas *Lainnya* hampir selalu tertangkap |
| **DINOv2 ViT-S/14** (`timm`, `lvd142m`) | dilatih *self-supervised* pada 142 juta gambar | 384-d | kuat di **tekstur & bentuk** (lipatan, sobekan) |

Kedua vektor dinormalisasi L2 (panjang = 1) lalu disambung: 512 + 384 = **896 dimensi** (fitur `base`).

Embedding dihitung **sekali** dan disimpan sebagai `.npy`, lalu dipakai ulang oleh banyak classifier.

In [ ]:
import open_clip
import timm


class ImgDS(Dataset):
    def __init__(self, rels, tf):
        self.rels, self.tf = rels, tf

    def __len__(self):
        return len(self.rels)

    def __getitem__(self, i):
        try:
            img = load_rgb(src_path(self.rels[i]))
            return self.tf(img), i, 1
        except Exception:
            return torch.zeros(3, 224, 224), i, 0


@torch.no_grad()
def extract(name, fwd, tf, bs=EMB_BS):
    for split, rels in SPLITS.items():
        out = EMB / f"emb_{name}_{split}.npy"
        if out.exists():
            print(f"[{name}] {split} sudah ada, dilewati"); continue
        dl = DataLoader(ImgDS(rels, tf), batch_size=bs, num_workers=NUM_WORKERS, shuffle=False)
        feats = None; t0 = time.time(); nbad = 0
        for x, idx, okk in dl:
            with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                f = fwd(x.to(DEV, non_blocking=True))
            f = f.float().cpu().numpy()
            if feats is None:
                feats = np.zeros((len(rels), f.shape[1]), np.float32)
            feats[idx.numpy()] = f; nbad += int((okk == 0).sum())
        np.save(out, feats)
        print(f"[{name}] {split} selesai {feats.shape} dalam {time.time() - t0:.0f}s, gagal dibaca: {nbad}")


def need(names):
    return not all((EMB / f"emb_{n}_{s}.npy").exists() for n in names for s in SPLITS)


CLIP_ARCH, CLIP_PRE = "ViT-B-16", "laion2b_s34b_b88k"
clip_model = None
if need(["clip"]) or RUN["zeroshot"] or (RUN["mlp_multicrop"] and need(["clipmc"])):
    clip_model, _, clip_tf = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRE)
    clip_model = clip_model.to(DEV).eval()

    def clip_fwd(x):
        f = clip_model.encode_image(x)
        return f / f.norm(dim=-1, keepdim=True)          # L2-normalisasi

    extract("clip", clip_fwd, clip_tf)

if need(["dino"]):
    dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224)
    dino = dino.to(DEV).eval()
    cfg = timm.data.resolve_data_config({}, model=dino); cfg["input_size"] = (3, 224, 224)
    extract("dino", lambda x: dino(x), timm.data.create_transform(**cfg))
    del dino; torch.cuda.empty_cache()

### Menyusun fitur `base`
- `l2(x)`: normalisasi L2 per baris (dengan `clip(1e-8)` agar tidak membagi nol).
- `base` = CLIP (sudah L2) ⊕ DINOv2 yang di-L2 dulu → 896-d.

In [ ]:
def l2(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True).clip(1e-8)


def load_emb(name):
    return {s: np.load(EMB / f"emb_{name}_{s}.npy") for s in SPLITS}


CLIP_E, DINO_E = load_emb("clip"), load_emb("dino")
BASE = {s: np.concatenate([CLIP_E[s], l2(DINO_E[s])], 1).astype(np.float32) for s in SPLITS}
print({s: v.shape for s, v in BASE.items()})

---
# §6 · Tahap 5: Classifier Non-linear di Atas Embedding (MLP)
Classifier linear (Logistic Regression) mentok di Macro-F1 ± 0,84 karena perbedaan Aman vs Rusak ada di detail halus yang tidak bisa dipisahkan dengan garis lurus. Dengan fitur yang **sama persis**, classifier non-linear (MLP/SVM) naik ke ± 0,91.

**MLP** (*Multi-Layer Perceptron*): `Dropout 0,1 → Linear d→512 → GELU → Dropout 0,3 → Linear 512→3`.

| Komponen | Fungsi |
|---|---|
| **StandardScaler** | tiap kolom fitur dibuat rata-rata 0, simpangan baku 1 (dihitung dari **train saja**) |
| **GELU** | aktivasi halus yang membuat jaringan non-linear |
| **Dropout** | mematikan sebagian neuron acak saat latihan agar tidak menghafal |
| **Label smoothing 0,1** | target "100 % Aman" dilunakkan; lebih tahan terhadap label noise (ada label konflik di data) |
| **Bobot kelas √balanced** | bobot kelas c = √(N / (3·n_c)); kompromi antara tanpa bobot dan *balanced* penuh (yang terlalu agresif ke Rusak) |
| **AdamW + OneCycleLR** | optimizer dengan weight decay, learning rate naik (10 % awal) lalu turun |
| **Early stopping** (sabar 12 epoch) | bobot epoch dengan Macro-F1 val terbaik disimpan |
| **3 seed dirata-rata** | menekan noise antar-inisialisasi |

> Catatan jujur: early stopping memakai Macro-F1 val, sehingga angka val sedikit optimistis (bias seleksi). Sebaran antar-seed (± 0,01) menunjukkan selisih kecil antar varian tidak signifikan.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d, h=512, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, 3))

    def forward(self, x):
        return self.net(x)


def train_mlp(Xtr, ytr, Xva, yva, Xte, seed, epochs=60, patience=12, lr=1e-3, wd=5e-2, bs=256, ls=0.1, **_):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr)
    Xva_t, Xte_t = torch.tensor(Xva), torch.tensor(Xte)
    cnt = np.bincount(ytr, minlength=3)
    w = torch.tensor((len(ytr) / (3 * cnt)) ** 0.5, dtype=torch.float32)       # bobot kelas sqrt-balanced
    model = MLP(Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(ytr) / bs)),
                                                pct_start=0.1)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=ls)
    best, bad = (-1, None, 0), 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); lossf(model(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            f1 = macro_f1(yva, model(Xva_t).argmax(1).numpy())
        if f1 > best[0]:
            best = (f1, {k: v.clone() for k, v in model.state_dict().items()}, ep); bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    print(f"  seed {seed}: berhenti di epoch {ep}, val Macro-F1 terbaik {best[0]:.4f} @ epoch {best[2]}")
    model.load_state_dict(best[1]); model.eval()
    with torch.no_grad():
        return torch.softmax(model(Xva_t), 1).numpy(), torch.softmax(model(Xte_t), 1).numpy()


def fit_mlp_ensemble(feats, name, cfg=MLP_CFG):
    sc = StandardScaler().fit(feats["train"])
    Xtr, Xva, Xte = (sc.transform(feats[s]).astype(np.float32) for s in ("train", "val", "test"))
    pv, pt = [], []
    for seed in cfg["seeds"]:
        a, b = train_mlp(Xtr, Y_TR, Xva, Y_VAL, Xte, seed, **cfg)
        pv.append(a); pt.append(b)
        print(f"    seed {seed} -> val penuh {macro_f1(Y_VAL, a.argmax(1)):.4f} | "
              f"bersih {macro_f1(Y_VAL[CLEAN_MASK], a[CLEAN_MASK].argmax(1)):.4f}")
    return write_contract(name, np.mean(pv, 0), np.mean(pt, 0), extra={"feat_dim": int(Xtr.shape[1])})


if RUN["mlp_base"]:
    if run_done("emb-heads-mlp"):
        print("emb-heads-mlp sudah ada, dilewati")
    else:
        t0 = time.time()
        fit_mlp_ensemble(BASE, "emb-heads-mlp")
        print(f"selesai {time.time() - t0:.0f}s")

---
# §7 · Tahap 6: Putaran 2 — CLIP Multi-crop 448 px
Kelemahan utama setelah putaran 1 adalah kelas **Rusak**: banyak foto Rusak masih ditebak Aman. Hipotesisnya: CLIP hanya menerima 224 px, sehingga detail kecil (penyok, sobekan) hilang ketika foto diperkecil.

**Multi-crop**: foto diperkecil ke 448 px (sisi pendek), dipotong tengah 448×448, lalu diambil **5 potongan 224×224** (empat kuadran + tengah). Tiap potongan diubah menjadi embedding CLIP sendiri; kelima vektor dirata-ratakan lalu dinormalisasi L2. Tiap bagian foto kini dilihat dalam resolusi 2× lebih tinggi.

```
gambar 448×448 → [kiri-atas, kanan-atas, kiri-bawah, kanan-bawah, tengah] → CLIP → 5×512 → rata-rata → L2 → 512-d
```
Fitur akhir `mc+base` = 512 + 896 = **1.408 dimensi**, lalu dilatih MLP 3 seed yang sama seperti §6.

In [ ]:
CLIP_MEAN, CLIP_STD = open_clip.OPENAI_DATASET_MEAN, open_clip.OPENAI_DATASET_STD


class MultiCropDS(Dataset):
    BOXES = [(0, 0, 224, 224), (224, 0, 448, 224), (0, 224, 224, 448), (224, 224, 448, 448), (112, 112, 336, 336)]

    def __init__(self, rels):
        self.rels = rels
        self.pre = T.Compose([T.Resize(448, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(448)])
        self.post = T.Compose([T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)])

    def __len__(self):
        return len(self.rels)

    def __getitem__(self, i):
        try:
            img = self.pre(load_rgb(src_path(self.rels[i])))
            return torch.stack([self.post(img.crop(b)) for b in self.BOXES]), i, 1   # (5, 3, 224, 224)
        except Exception:
            return torch.zeros(5, 3, 224, 224), i, 0


@torch.no_grad()
def extract_multicrop(bs=32):
    for split, rels in SPLITS.items():
        out = EMB / f"emb_clipmc_{split}.npy"
        if out.exists():
            print(f"[clipmc] {split} sudah ada, dilewati"); continue
        dl = DataLoader(MultiCropDS(rels), batch_size=bs, num_workers=NUM_WORKERS, shuffle=False)
        feats = np.zeros((len(rels), 5, 512), np.float32); t0 = time.time()
        for x, idx, _ in dl:
            B = x.shape[0]
            with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                f = clip_model.encode_image(x.view(-1, 3, 224, 224).to(DEV, non_blocking=True))
            f = f.float(); f = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy().reshape(B, 5, -1)
            feats[idx.numpy()] = f
        np.save(out, feats)
        print(f"[clipmc] {split} selesai {feats.shape} dalam {time.time() - t0:.0f}s")


if RUN["mlp_multicrop"]:
    if need(["clipmc"]):
        if clip_model is None:
            clip_model, _, clip_tf = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRE)
            clip_model = clip_model.to(DEV).eval()
        extract_multicrop()
    MC = {s: l2(np.load(EMB / f"emb_clipmc_{s}.npy").mean(1)) for s in SPLITS}
    MC_BASE = {s: np.concatenate([MC[s], BASE[s]], 1).astype(np.float32) for s in SPLITS}
    if run_done("emb-r2-mlp"):
        print("emb-r2-mlp sudah ada, dilewati")
    else:
        t0 = time.time()
        fit_mlp_ensemble(MC_BASE, "emb-r2-mlp")
        print(f"selesai {time.time() - t0:.0f}s")

if not RUN["zeroshot"]:
    clip_model = None; torch.cuda.empty_cache()

---
# §8 · Tahap 7: Fine-tune ConvNeXt-tiny
Berbeda dengan §5–§7 yang membekukan model, di sini (hampir) seluruh bobot jaringan **dilatih ulang** pada foto paket. Lebih mahal, tetapi jaringan bisa menyesuaikan diri, dan memberi **keragaman** untuk ensemble.

**ConvNeXt-tiny** (`convnext_tiny.fb_in22k_ft_in1k`, 27,8 juta parameter): CNN modern yang meniru resep Vision Transformer (kernel 7×7, LayerNorm, GELU). Pretrain di ImageNet-22k lalu fine-tune di ImageNet-1k.

| Teknik | Apa itu | Gunanya di sini |
|---|---|---|
| **Transfer learning** | bobot pretrained dipakai, *head* diganti 3 kelas | dataset kecil dan kotor; dari nol akan overfit |
| **WeightedRandomSampler** | peluang ambil foto ∝ n_kelas^(−p) | *Lainnya* muncul lebih sering tiap epoch (p = 0,75 → ± 45/34/21 %; p = 0,5 → ± 55/32/12 %) |
| **Label smoothing 0,1** | target 1,0 → ± 0,93 | tahan terhadap label noise |
| **AdamW + weight decay** | Adam dengan peluruhan bobot yang benar | mengurangi overfit |
| **LR body 1e-4, head 1e-3** | head baru belajar 10× lebih cepat | body pretrained hanya disetel halus |
| **Warmup 1 epoch + cosine** | LR naik pelan lalu turun mengikuti kosinus | stabil di awal, halus di akhir |
| **Mixed precision** | forward dalam fp16/bf16, loss diskalakan (GradScaler) | lebih cepat & hemat memori |
| **drop_path 0,1** | melewati blok jaringan acak saat latihan | regularisasi |
| **TTA hflip** | rata-rata prediksi foto asli + foto dibalik horizontal | prediksi lebih stabil |
| **Checkpoint terbaik** | `best.pt` disimpan tiap kali Macro-F1 val naik | menghindari epoch yang sudah overfit |

## Dua putaran

| Item | Putaran 1 | Putaran 2 | Alasan |
|---|---|---|---|
| Resolusi | 192 | 224 | detail lebih banyak |
| Freeze | stem + 2 stage | tidak ada | kapasitas penuh |
| Augmentasi | RRC 0,6–1; jitter 0,3; grayscale 5 %; erasing 15 % | RRC 0,7–1; hflip; jitter 0,2/0,2/0,1 saja | buang pemicu jalan pintas (noise/rotasi) |
| Eval transform | Resize 1,14× + CenterCrop | Letterbox | seluruh gambar terlihat |
| Sampler | n^−0,75 | n^−0,5 | koreksi over-predict Rusak |
| Epoch | 5 | 7 | |

Training menyimpan `last.pt` setiap epoch: bila Colab putus, **Run all** akan melanjutkan dari epoch terakhir.

In [ ]:
class FocalLoss(nn.Module):
    # focal loss multi-kelas: -alpha_y * (1 - p_y)^gamma * log p_y  (dipakai EfficientNet di §11)
    def __init__(self, alpha, gamma=2.0, ls=0.0):
        super().__init__()
        self.register_buffer("alpha", torch.as_tensor(alpha, dtype=torch.float32)); self.gamma = gamma

    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), 1).gather(1, y[:, None]).squeeze(1)
        return (self.alpha[y] * (1 - logp.exp()).pow(self.gamma) * -logp).mean()


def make_tf(kind, size):
    norm = [T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)]
    if kind == "r1":
        return T.Compose([T.RandomResizedCrop(size, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                          T.ColorJitter(0.3, 0.3, 0.2), T.RandomGrayscale(0.05), *norm, T.RandomErasing(p=0.15)])
    if kind == "r2":
        return T.Compose([T.RandomResizedCrop(size, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                          T.ColorJitter(0.2, 0.2, 0.1), *norm])
    if kind == "squash_aug":
        return T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.1), *norm])
    if kind == "crop":
        return T.Compose([T.Resize(int(size * 1.14)), T.CenterCrop(size), *norm])
    if kind == "letterbox":
        return T.Compose([Letterbox(size), *norm])
    if kind == "squash":
        return T.Compose([T.Resize((size, size)), *norm])
    raise ValueError(kind)


@torch.no_grad()
def predict(model, dl, tta=True):
    model.eval(); probs, oks = [], []
    for x, _, okk in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            p = model(x).float().softmax(1)
            if tta:
                p = (p + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2
        probs.append(p.cpu()); oks.append(okk)
    P = torch.cat(probs).numpy(); okk = torch.cat(oks).numpy()
    P[okk == 0] = 1.0 / 3                      # file tak terbaca -> probabilitas seragam
    return P


def train_cnn(name, cfg):
    out = RUNS / name; out.mkdir(parents=True, exist_ok=True)
    if run_done(name):
        print(f"[{name}] sudah selesai, dilewati"); return json.load(open(out / "metrics.json"))
    seed_everything()
    size = cfg["img"]
    counts = np.bincount(Y_TR, minlength=3)
    ds_tr = CsvDataset(TRAIN_REL, Y_TR, make_tf(cfg["train_aug"], size), size, train=True)
    ds_va = CsvDataset(VAL_REL, Y_VAL, make_tf(cfg["eval_tf"], size), size)
    ds_te = CsvDataset(TEST_REL, None, make_tf(cfg["eval_tf"], size), size)

    # sampler: bobot tiap foto = count[kelas]^-p (bukan cross-entropy berbobot)
    cls_w = counts.astype(float) ** (-cfg["sampler_power"])
    eff = cls_w * counts / (cls_w * counts).sum()
    print(f"[{name}] sampler p={cfg['sampler_power']} -> porsi kelas per epoch {dict(zip(CLASSES, np.round(eff, 3)))}")
    sampler = WeightedRandomSampler(torch.as_tensor(cls_w[Y_TR], dtype=torch.double), len(Y_TR), replacement=True)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=DEV.type == "cuda", persistent_workers=NUM_WORKERS > 0)
    dl_tr = DataLoader(ds_tr, batch_size=cfg["bs"], sampler=sampler, drop_last=True, **kw)
    dl_va = DataLoader(ds_va, batch_size=96, shuffle=False, **kw)

    model = timm.create_model(cfg["model"], pretrained=True, num_classes=3, drop_rate=cfg["drop_rate"],
                              drop_path_rate=cfg["drop_path"]).to(DEV)
    if cfg.get("freeze_stages", 0) > 0 and hasattr(model, "stages"):
        for mod in [model.stem] + list(model.stages[:cfg["freeze_stages"]]):
            for p in mod.parameters():
                p.requires_grad = False
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids]
    body_params = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    n_train = sum(p.numel() for p in body_params + head_params)
    print(f"[{name}] parameter dilatih {n_train / 1e6:.2f}M / {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    opt = torch.optim.AdamW([{"params": body_params, "lr": cfg["lr"]}, {"params": head_params, "lr": cfg["head_lr"]}],
                            weight_decay=cfg["wd"])
    spe = len(dl_tr); total = spe * cfg["epochs"]; warm = spe          # warmup 1 epoch

    def lr_lambda(s):
        if s < warm:
            return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    if cfg["loss"] == "focal":
        alpha = counts.astype(float) ** -0.5; alpha = alpha / alpha.mean()
        crit = FocalLoss(alpha, gamma=2.0).to(DEV)
    else:
        crit = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

    best, hist, start_ep = -1.0, [], 0
    if (out / "last.pt").exists():                                      # lanjut setelah sesi putus
        ck = torch.load(out / "last.pt", map_location=DEV, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); best, hist, start_ep = ck["best"], ck["hist"], ck["epoch"] + 1
        print(f"[{name}] melanjutkan dari epoch {start_ep + 1} (best F1 sejauh ini {best:.4f})")

    t_all = time.time()
    for ep in range(start_ep, cfg["epochs"]):
        model.train(); te = time.time(); run, n = 0.0, 0
        for x, y, _ in dl_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = crit(model(x), y)
            if not torch.isfinite(loss):                                 # batch NaN/inf dilewati
                print("  loss non-finite, batch dilewati"); opt.zero_grad(set_to_none=True); sched.step(); continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if cfg.get("grad_clip"):
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); n += 1
        pva = predict(model, dl_va, tta=False)
        f1 = macro_f1(Y_VAL, pva.argmax(1))
        pc = f1_score(Y_VAL, pva.argmax(1), average=None, labels=[0, 1, 2])
        hist.append({"epoch": ep + 1, "loss": run / max(n, 1), "macro_f1": float(f1), "per_class_f1": pc.tolist(),
                     "sec": time.time() - te})
        print(f"[{name}] epoch {ep + 1}/{cfg['epochs']} loss {run / max(n, 1):.4f} | val Macro-F1 {f1:.4f} | "
              f"per kelas {np.round(pc, 3).tolist()} | {(time.time() - te) / 60:.1f} menit")
        if f1 > best:
            best = float(f1); torch.save({"model": model.state_dict(), "epoch": ep + 1, "macro_f1": best}, out / "best.pt")
            print(f"    simpan best.pt (epoch {ep + 1})")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "best": best, "hist": hist, "epoch": ep}, out / "last.pt")
        json.dump({"history": hist, "best_macro_f1": best}, open(out / "history.json", "w"), indent=1)
        if cfg.get("max_minutes") and (time.time() - t_all) / 60 > cfg["max_minutes"]:
            print(f"[{name}] batas waktu {cfg['max_minutes']} menit tercapai, training dihentikan"); break
    print(f"[{name}] training selesai, best val Macro-F1 (tanpa TTA) {best:.4f}")

    ck = torch.load(out / "best.pt", map_location=DEV, weights_only=False)
    model.load_state_dict(ck["model"])
    dl_te = DataLoader(ds_te, batch_size=96, shuffle=False, **kw)
    pva, pte = predict(model, dl_va, cfg["tta"]), predict(model, dl_te, cfg["tta"])
    m = write_contract(name, pva, pte, extra={"best_epoch": ck["epoch"], "cfg": cfg})
    (out / "last.pt").unlink(missing_ok=True)                          # hemat ruang Drive
    del model, opt; torch.cuda.empty_cache()
    return m


def plot_history(name):
    h = json.load(open(RUNS / name / "history.json"))["history"]
    ep = [r["epoch"] for r in h]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(ep, [r["loss"] for r in h], "o-"); ax[0].set_title(f"{name} · train loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(ep, [r["macro_f1"] for r in h], "o-", label="Macro-F1")
    for k, c in enumerate(CLASSES):
        ax[1].plot(ep, [r["per_class_f1"][k] for r in h], "--", alpha=0.7, label=f"F1 {c}")
    ax[1].set_title(f"{name} · val (tanpa TTA)"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
for _name, _key in [("convnext-r1", "convnext_r1"), ("convnext-r2", "convnext_r2")]:
    if RUN[_key]:
        t0 = time.time()
        train_cnn(_name, CNN_CFGS[_name])
        print(f"[{_name}] total {(time.time() - t0) / 60:.1f} menit\n")
        if (RUNS / _name / "history.json").exists():
            plot_history(_name)

---
# §9 · Tahap 8: Ensemble dan Pasca-proses
Setiap model punya kekuatan berbeda: embedding + MLP sangat baik untuk *Lainnya*, ConvNeXt lebih baik membaca detail Aman/Rusak. Ensemble menggabungkannya.

## 9.1 Rata-rata log-probabilitas berbobot + bias logit per kelas
$$\text{skor}_\text{kelas} = \frac{\sum_m w_m \cdot \log(p_m + 10^{-6})}{\sum_m w_m} + b_\text{kelas}$$

| | |
|---|---|
| **Kenapa log** | rata-rata geometrik menghukum model yang sangat yakin tetapi salah, dan cocok dengan bias logit sesudahnya |
| **Bias logit** `[0, b_Rusak, b_Lainnya]` | menggeser ambang tiap kelas untuk memaksimalkan Macro-F1 langsung (model dengan bobot kelas cenderung over-predict kelas kecil) |
| **Pencarian** | bobot tiap model dicoba 0–3 (0 = tidak dipakai); `b_Lainnya` −2..2 dan `b_Rusak` −2..1,5 langkah 0,25 |
| **Dinilai di** | **val bersih** saja, supaya tidak tertipu foto kembar |

Probabilitas tiap model digabung berdasarkan `path` (val) dan `filename` (test), sehingga urutan baris model tidak perlu sama. Foto yang tidak ada di suatu model diberi probabilitas seragam 1/3.

In [ ]:
def load_runs(names):
    V, Tt = {}, {}
    Pc = ["p_Aman", "p_Rusak", "p_Lainnya"]
    for r in names:
        v = pd.read_csv(RUNS / r / "val_probs.csv"); t = pd.read_csv(RUNS / r / "test_probs.csv")
        v = VAL[["path"]].merge(v[["path"] + Pc], on="path", how="left")
        t = pd.DataFrame({"filename": TEST_FILES}).merge(t[["filename"] + Pc], on="filename", how="left")
        mv, mt = v[Pc].isna().any(axis=1).sum(), t[Pc].isna().any(axis=1).sum()
        V[r] = v[Pc].fillna(1 / 3).values; Tt[r] = t[Pc].fillna(1 / 3).values
        pr = V[r].argmax(1)
        print(f"{r:16s} val F1 {macro_f1(Y_VAL, pr):.4f} | BERSIH F1 {macro_f1(Y_VAL[CLEAN_MASK], pr[CLEAN_MASK]):.4f} | "
              f"per kelas (bersih) {np.round(f1_score(Y_VAL[CLEAN_MASK], pr[CLEAN_MASK], average=None, labels=[0, 1, 2]), 3)} | "
              f"hilang {mv}/{mt}")
    return V, Tt


def comb(D, w, names):
    return sum(wi * np.log(D[k] + 1e-6) for wi, k in zip(w, names)) / sum(w)


names = [r for r in ENSEMBLE_RUNS if run_done(r)]
missing = [r for r in ENSEMBLE_RUNS if r not in names]
if missing:
    print("PERINGATAN: run berikut belum ada dan tidak ikut ensemble:", missing)
assert names, "belum ada model yang selesai"
V, Tt = load_runs(names)

yc = Y_VAL[CLEAN_MASK]
grid = [0, 1, 2, 3] if len(names) <= 4 else [0, 1, 2]
B1 = np.arange(-2.0, 1.51, 0.25); B2 = np.arange(-2.0, 2.01, 0.25)
best = (-1, None, None); t0 = time.time()
for w in itertools.product(grid, repeat=len(names)):
    if sum(w) == 0:
        continue
    lv = comb(V, w, names)[CLEAN_MASK]
    for b2 in B2:
        for b1 in B1:
            f = macro_f1(yc, (lv + np.array([0, b1, b2])).argmax(1))
            if f > best[0] + 1e-12:
                best = (f, w, (b1, b2))
f_best, W, (b1, b2) = best
BIAS = np.array([0, b1, b2])
print(f"\npencarian selesai dalam {time.time() - t0:.0f}s")
print(f"TERBAIK di val BERSIH: Macro-F1 {f_best:.4f} | bobot {dict(zip(names, W))} | bias Rusak {b1:+.2f}, Lainnya {b2:+.2f}")
PV = comb(V, W, names) + BIAS
ENS_METRICS = metrics(Y_VAL, PV, CLEAN_MASK)
show_metrics("ENSEMBLE", ENS_METRICS)

## 9.2 Prediksi test: rata-rata antar foto kembar + (opsional) override pseudo-label
1. Skor gabungan test dihitung dengan bobot yang sama.
2. **Rata-rata kembar**: skor dirata-ratakan di dalam tiap grup foto test yang kembar (§3.4), sehingga satu foto yang kebetulan dinilai salah tidak berbeda dari kembarannya.
3. `argmax(skor + bias)` → label.
4. **Override** (hanya bila `USE_PSEUDO_OVERRIDE = True`): foto test yang identik dengan foto train berlabel konsisten langsung diberi label train. Default **mati**: memanfaatkan kebocoran train–test hanya boleh bila aturan lomba mengizinkan.

In [ ]:
LT = comb(Tt, W, names)
if TWIN_AVERAGE:
    LT = pd.DataFrame(LT).groupby(TEST_GRP).transform("mean").values
pt = (LT + BIAS).argmax(1)
OUT = pd.DataFrame({"filename": TEST_FILES, "label": [CLASSES[i] for i in pt]})

if USE_PSEUDO_OVERRIDE:
    m_ = OUT.merge(PSEUDO.rename(columns={"test_filename": "filename", "label": "pl"}), on="filename", how="left")
    chg = (m_.pl.notna() & (m_.pl != m_.label)).sum()
    OUT["label"] = np.where(m_.pl.notna(), m_.pl, m_.label)
    print(f"override pseudo-label: {m_.pl.notna().sum()} file, {chg} berubah")
else:
    m_ = OUT.merge(PSEUDO.rename(columns={"test_filename": "filename", "label": "pl"}), on="filename", how="inner")
    print(f"override dimatikan. (Info: dari {len(m_)} foto test yang identik dengan train, "
          f"prediksi model sama dengan label train pada {(m_.pl == m_.label).mean():.1%})")

SUB = SAMPLE[["filename"]].merge(OUT, on="filename", how="left")
assert len(SUB) == len(SAMPLE) and SUB.label.notna().all() and SUB.label.isin(CLASSES).all()
assert (SUB.filename.values == SAMPLE.filename.values).all()
(RUNS / "ensemble").mkdir(exist_ok=True)
SUB.to_csv(RUNS / "ensemble" / "submission.csv", index=False)
SUB.to_csv(WORK / "submission.csv", index=False)
pd.DataFrame({"filename": TEST_FILES, "s_Aman": LT[:, 0] + BIAS[0], "s_Rusak": LT[:, 1] + BIAS[1],
              "s_Lainnya": LT[:, 2] + BIAS[2]}).to_csv(RUNS / "ensemble" / "test_scores.csv", index=False)
json.dump({"runs": names, "weights": list(map(int, W)), "bias": BIAS.tolist(), "val": ENS_METRICS,
           "override": USE_PSEUDO_OVERRIDE, "twin_average": TWIN_AVERAGE},
          open(RUNS / "ensemble" / "ensemble.json", "w"), indent=1)
print("submission ditulis ke", WORK / "submission.csv")
print("distribusi label test:", SUB.label.value_counts().to_dict(), "| baris:", len(SUB))
display(SUB.head())

---
# §10 · Hasil Akhir dan Perbandingan
Tabel di bawah membaca `metrics.json` setiap run yang ada (termasuk eksperimen opsional di §11 bila dijalankan).

**Cara membaca angka ini dengan benar**
- *Val bersih* lebih jujur daripada val penuh, tetapi tetap bukan test: sebagian kelompok foto test (render AI berukuran besar, foto 224×224 ber-noise) hampir tidak terwakili di val.
- Bobot ensemble dan bias dicari di val bersih, sehingga angka ensemble sedikit optimistis (bias seleksi). Selisih < ± 0,01 antar varian teratas masih dalam noise antar-seed.
- Hanya ± 60 foto *Lainnya* di val bersih: satu-dua foto salah sudah menggeser F1 Lainnya beberapa persen.

In [ ]:
rows = []
for d in sorted(RUNS.iterdir()):
    if (d / "metrics.json").exists():
        m = json.load(open(d / "metrics.json"))
        rows.append(dict(run=d.name, val_penuh=m["macro_f1"], val_bersih=m["macro_f1_clean"],
                         **{f"F1 {c} (bersih)": m["per_class_f1_clean"][c] for c in CLASSES}))
rows.append(dict(run="ENSEMBLE", val_penuh=ENS_METRICS["macro_f1"], val_bersih=ENS_METRICS["macro_f1_clean"],
                 **{f"F1 {c} (bersih)": ENS_METRICS["per_class_f1_clean"][c] for c in CLASSES}))
RES = pd.DataFrame(rows).sort_values("val_bersih").reset_index(drop=True)
display(RES.style.format({c: "{:.4f}" for c in RES.columns if c != "run"}))

fig, ax = plt.subplots(1, 2, figsize=(13, 0.45 * len(RES) + 1.5), gridspec_kw={"width_ratios": [1.4, 1]})
y = np.arange(len(RES))
ax[0].hlines(y, RES.val_bersih, RES.val_penuh, color="#9db4d6")
ax[0].scatter(RES.val_penuh, y, facecolors="white", edgecolors="#2f6db5", label="val penuh", zorder=3)
ax[0].scatter(RES.val_bersih, y, color="#2f6db5", label="val bersih", zorder=3)
ax[0].set_yticks(y); ax[0].set_yticklabels(RES.run); ax[0].set_xlabel("Macro-F1"); ax[0].legend(loc="lower right")
ax[0].set_title("Macro-F1 tiap pendekatan")
cm = np.array(ENS_METRICS["cm_clean"])
ax[1].imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(CLASSES); ax[1].set_yticks(range(3)); ax[1].set_yticklabels(CLASSES)
ax[1].set_xlabel("prediksi"); ax[1].set_ylabel("label asli"); ax[1].set_title("Confusion matrix ensemble (val bersih)")
plt.tight_layout(); plt.show()

print(classification_report(Y_VAL[CLEAN_MASK], PV[CLEAN_MASK].argmax(1), target_names=CLASSES, digits=4))

In [ ]:
# distribusi prediksi test tiap model vs ensemble
dist = {}
for d in sorted(RUNS.iterdir()):
    if (d / "submission.csv").exists():
        dist[d.name] = pd.read_csv(d / "submission.csv").label.value_counts().reindex(CLASSES).fillna(0).astype(int)
DIST = pd.DataFrame(dist).T
display(DIST)
print("porsi kelas di train:", (TRAIN.label.value_counts(normalize=True).reindex(CLASSES).round(3)).to_dict())

# contoh foto val bersih yang salah ditebak ensemble
wrong = np.where(CLEAN_MASK & (PV.argmax(1) != Y_VAL))[0]
if len(wrong):
    pick = np.random.RandomState(0).choice(wrong, min(10, len(wrong)), replace=False)
    show_grid([DATA_ROOT / VAL_REL[i] for i in pick],
              [f"asli {CLASSES[Y_VAL[i]]} → {CLASSES[PV[i].argmax()]}" for i in pick])

---
# §11 · Eksperimen Pembanding (opsional)
Bagian ini **tidak dibutuhkan** untuk `submission.csv` final. Nyalakan saklarnya di `RUN` (§0.1) bila ingin membandingkan. Setiap eksperimen menulis *file contract* ke `runs/<nama>/`, sehingga otomatis muncul di tabel §10. Untuk ikut ensemble, tambahkan namanya ke `ENSEMBLE_RUNS` lalu jalankan ulang §9.

| Eksperimen | Saklar | Hasil di laporan (val penuh) |
|---|---|---|
| Zero-shot CLIP (tanpa training, hanya kalimat deskripsi) | `zeroshot` | 0,33 — gagal |
| Logistic Regression [CLIP + DINOv2] | `logreg` | 0,84 |
| kNN cosine (k = 5, 15) | `knn` | 0,85–0,88 |
| SVM RBF [CLIP + DINOv2], C = 10, Platt scaling | `svm` | 0,91 |
| CLIP ViT-L/14 + fitur base, SVM C = 30 | `vit_l14` | 0,92 |
| EfficientNetV2-b0 + focal loss | `effnet_focal` | 0,83 |
| YOLO11s-cls (Ultralytics) | `yolo` | 0,81 |

In [ ]:
# ---- 11.1 Zero-shot CLIP ----------------------------------------------------
if RUN["zeroshot"] and not run_done("zeroshot-clip"):
    if clip_model is None:
        clip_model, _, clip_tf = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRE)
        clip_model = clip_model.to(DEV).eval()
    tok = open_clip.get_tokenizer(CLIP_ARCH)
    prompts = {"Aman": ["a photo of an intact cardboard shipping box",
                        "a photo of an undamaged cardboard parcel box in good condition"],
               "Rusak": ["a photo of a damaged, dented, torn or wet cardboard box",
                         "a photo of a crushed, broken or ripped cardboard parcel"],
               "Lainnya": ["a photo of a plastic shopping bag, paper bag or food packaging",
                           "a photo of a plastic bag, paper bag, pouch or food package, not a cardboard box"]}
    with torch.no_grad():
        Tm = []
        for c in CLASSES:
            t = clip_model.encode_text(tok(prompts[c]).to(DEV)).float(); t = t / t.norm(dim=-1, keepdim=True)
            t = t.mean(0); Tm.append((t / t.norm()).cpu().numpy())
    Tm = np.stack(Tm)
    zs = {s: torch.softmax(torch.tensor(100 * CLIP_E[s] @ Tm.T), 1).numpy() for s in ("val", "test")}
    write_contract("zeroshot-clip", zs["val"], zs["test"])

# ---- 11.2 Logistic Regression, kNN, SVM di atas fitur base ------------------
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

if any(RUN[k] for k in ("logreg", "knn", "svm")):
    sc_b = StandardScaler().fit(BASE["train"])
    Xb = {s: sc_b.transform(BASE[s]) for s in SPLITS}
if RUN["logreg"] and not run_done("fm-probe-logreg"):
    best_lr = None
    for C in [0.01, 0.03, 0.1, 1, 10]:
        clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(Xb["train"], Y_TR)
        f = macro_f1(Y_VAL, clf.predict(Xb["val"]))
        print(f"  LR C={C}: val Macro-F1 {f:.4f}")
        if best_lr is None or f > best_lr[0]:
            best_lr = (f, clf, C)
    write_contract("fm-probe-logreg", best_lr[1].predict_proba(Xb["val"]), best_lr[1].predict_proba(Xb["test"]),
                   extra={"C": best_lr[2]})
if RUN["knn"]:
    for k in (5, 15):
        if not run_done(f"emb-heads-knn{k}"):
            knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", weights="distance").fit(BASE["train"], Y_TR)
            write_contract(f"emb-heads-knn{k}", knn.predict_proba(BASE["val"]), knn.predict_proba(BASE["test"]))
if RUN["svm"] and not run_done("emb-heads-svm"):
    t0 = time.time()
    svm = SVC(C=10, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=0)
    svm.fit(Xb["train"], Y_TR)
    write_contract("emb-heads-svm", svm.predict_proba(Xb["val"]), svm.predict_proba(Xb["test"]), extra={"C": 10})
    print(f"  SVM selesai {time.time() - t0:.0f}s")

In [ ]:
# ---- 11.3 CLIP ViT-L/14 + fitur base, SVM C=30 ---------------------------------
if RUN["vit_l14"]:
    if need(["l14"]):
        l14, _, l14_tf = open_clip.create_model_and_transforms("ViT-L-14", pretrained="laion2b_s32b_b82k")
        l14 = l14.to(DEV).eval()

        def l14_fwd(x):
            f = l14.encode_image(x); return f / f.norm(dim=-1, keepdim=True)

        extract("l14", l14_fwd, l14_tf, bs=32)
        del l14; torch.cuda.empty_cache()
    if not run_done("emb-r2-svm-l14"):
        L14 = load_emb("l14")
        X = {s: np.concatenate([L14[s], BASE[s]], 1) for s in SPLITS}
        sc_l = StandardScaler().fit(X["train"]); X = {s: sc_l.transform(X[s]) for s in SPLITS}
        t0 = time.time()
        svm = SVC(C=30, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=0)
        svm.fit(X["train"], Y_TR)
        write_contract("emb-r2-svm-l14", svm.predict_proba(X["val"]), svm.predict_proba(X["test"]), extra={"C": 30})
        print(f"  SVM[l14+base] selesai {time.time() - t0:.0f}s")

# ---- 11.4 EfficientNetV2-b0 + focal loss ----------------------------------------
if RUN["effnet_focal"]:
    EFF_CFG = dict(model="tf_efficientnetv2_b0.in1k", img=224, epochs=6, bs=48, lr=3e-4, head_lr=3e-3, wd=0.02,
                   loss="focal", label_smoothing=0.0, drop_rate=0.2, drop_path=0.1, grad_clip=2.0, tta=True,
                   freeze_stages=0, sampler_power=0.5, train_aug="squash_aug", eval_tf="squash", max_minutes=None)
    train_cnn("effnet-focal", EFF_CFG)

### 11.5 YOLO11s-cls
Ultralytics membaca dataset berbentuk folder `train/{kelas}/` dan `val/{kelas}/`. Folder dibuat dari split memakai *symlink* ke cache (tanpa menyalin), dan foto *Lainnya* di train digandakan 4× (*oversampling*). Urutan kelas internal Ultralytics alfabetis (Aman, Lainnya, Rusak), sehingga dipetakan ulang sebelum menulis probabilitas.

In [ ]:
if RUN["yolo"] and not run_done("yolo-cls"):
    %pip install -q ultralytics
    from ultralytics import YOLO
    YDS = LOCAL_DIR / "yolo_ds"
    if YDS.exists():
        shutil.rmtree(YDS)
    for split, df_ in (("train", TRAIN), ("val", VAL)):
        for c in CLASSES:
            (YDS / split / c).mkdir(parents=True, exist_ok=True)
        for p, lab in zip(df_.path, df_.label):
            src = src_path(p); reps = 4 if (split == "train" and lab == "Lainnya") else 1
            for r in range(reps):
                stem = Path(p).name.replace(".", "_")
                dst = YDS / split / lab / (f"{stem}.jpg" if r == 0 else f"{stem}__dup{r}.jpg")
                if not dst.exists():
                    os.symlink(src, dst)
    ymodel = YOLO("yolo11s-cls.pt")
    ymodel.train(data=str(YDS), imgsz=320, epochs=6, patience=3, batch=64, device=0 if DEV.type == "cuda" else "cpu",
                 workers=NUM_WORKERS, project=str(RUNS / "yolo-cls" / "ul"), name="run", exist_ok=True,
                 pretrained=True, seed=SEED, deterministic=False, plots=False, verbose=False,
                 fliplr=0.5, flipud=0.0, erasing=0.2, auto_augment="randaugment")
    best_w = RUNS / "yolo-cls" / "ul" / "run" / "weights" / "best.pt"
    ymodel = YOLO(str(best_w))
    order = [list(ymodel.names.values()).index(c) for c in CLASSES]

    def yolo_one(p):
        try:
            return ymodel.predict(p, imgsz=320, verbose=False)[0].probs.data.cpu().numpy()[order]
        except Exception:
            return np.full(3, 1 / 3, np.float32)            # file tak terbaca -> seragam

    def yolo_probs(rels):
        out = []
        for s in range(0, len(rels), 256):
            chunk = [str(src_path(r)) for r in rels[s:s + 256]]
            try:
                out += [r.probs.data.cpu().numpy()[order] for r in ymodel.predict(chunk, imgsz=320, verbose=False)]
            except Exception:
                out += [yolo_one(p) for p in chunk]
        return np.stack(out)

    write_contract("yolo-cls", yolo_probs(VAL_REL), yolo_probs(TEST_REL))

---
# §12 · Catatan
**Yang perlu dicek sebelum submit**
1. **Model pretrained**: CLIP, DINOv2, dan ConvNeXt memakai bobot yang diunduh dari internet. Pastikan aturan lomba mengizinkan.
2. **Kebocoran train–test**: override pseudo-label default **mati**. Rata-rata antar foto kembar di sisi test saja hanya memakai prediksi model (bukan label train).
3. **Angka val ≠ skor leaderboard**: val bersih adalah perkiraan terbaik, tetapi distribusi test berbeda dari train.

**Ide lanjutan**
- SVM di atas fitur multi-crop, dan multi-crop untuk DINOv2.
- Fine-tune ConvNeXt di 256–288 px.
- Membersihkan label konflik di train (grup md5 identik berlabel beda + pasangan near-dup lintas label, lihat `runs/eda/train_cross_label_pairs.csv`) sebelum melatih ulang.
- Sampler n^−0,6 atau ambang `p_Lainnya ≥ 0,3` untuk ConvNeXt putaran 2.